<a href="https://colab.research.google.com/github/penajuanmanuel6-hub/automated-etl-pipeline/blob/main/analisis_sismos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sqlite3
import pandas as pd
import folium
from folium.plugins import HeatMap

# 1. CREAR UNA BASE DE DATOS SQL EN MEMORIA Y CARGAR DATOS SIMULADOS
# Simulamos una base de datos corporativa con activos de infraestructura crítica (Oleoductos/Plantas)
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# Creamos la tabla de activos energéticos
cursor.execute('''
    CREATE TABLE activos_infraestructura (
        id_activo TEXT,
        nombre_activo TEXT,
        tipo TEXT,
        latitud REAL,
        longitud REAL,
        departamento TEXT
    )
''')

# Insertamos datos de activos reales ubicados en la zona de influencia del sismo (Chocó, Valle, Risaralda, Antioquia)
activos = [
    ('ACT-01', 'Planta de Bombeo Istmina', 'Bombeo Crudo', 5.15, -76.67, 'Chocó'),
    ('ACT-02', 'Subestación Yumbo', 'Energía Eléctrica', 3.58, -76.51, 'Valle del Cauca'),
    ('ACT-03', 'Refinería de Cartago', 'Refinación', 4.74, -75.91, 'Valle del Cauca'),
    ('ACT-04', 'Nodo Logístico Pereira', 'Almacenamiento', 4.81, -75.69, 'Risaralda'),
    ('ACT-05', 'Planta Termoeléctrica Dorada', 'Generación', 5.45, -74.66, 'Caldas'),
    ('ACT-06', 'Estación de Gas Yotoco', 'Gasoducto', 3.87, -76.38, 'Valle del Cauca')
]

cursor.executemany('INSERT INTO activos_infraestructura VALUES (?, ?, ?, ?, ?, ?)', activos)
conn.commit()

# 2. CONSULTAR LA BASE DE DATOS USANDO SQL
# Extraemos los activos para cruzarlos con el modelo de riesgo
query = "SELECT * FROM activos_infraestructura WHERE departamento IN ('Chocó', 'Valle del Cauca', 'Risaralda', 'Caldas')"
df_activos = pd.read_sql_query(query, conn)

print("--- ACTIVOS EXTRAÍDOS DE LA BASE DE DATOS SQL ---")
print(df_activos[['id_activo', 'nombre_activo', 'tipo', 'departamento']])
print("-" * 50)

# 3. CREAR EL MAPA GEOESPACIAL DE IMPACTO (DASHBOARD VISUAL)
# Centro geográfico aproximado del epicentro del sismo (San José del Palmar, Chocó)
lat_epicentro = 5.00
lon_epicentro = -76.50

# Inicializamos el mapa centrado en el occidente colombiano
mapa_crisis = folium.Map(location=[lat_epicentro, lon_epicentro], zoom_start=8)

# Añadir el marcador del Epicentro del Terremoto (Magnitud 7.4)
folium.Marker(
    location=[lat_epicentro, lon_epicentro],
    popup="<b>EPICENTRO TERREMOTO</b><br>Magnitud: 7.4<br>Fecha: 10 Ago 2026",
    icon=folium.Icon(color='red', icon='bolt', prefix='fa')
).add_to(mapa_crisis)

# Añadir un círculo de radio de afectación crítica (ej. 80 km a la redonda)
folium.Circle(
    location=[lat_epicentro, lon_epicentro],
    radius=80000, # 80 km en metros
    color='crimson',
    fill=True,
    fill_color='red',
    fill_opacity=0.15,
    tooltip="Zona de Alto Impacto Sísmico"
).add_to(mapa_crisis)

# 4. ITERAR SOBRE LOS ACTIVOS Y AGREGARLOS AL MAPA CON ESTADO OPERATIVO
for index, row in df_activos.iterrows():
    # Lógica simple: Si está en Chocó o Valle, se marca como inspección urgente por cercanía al epicentro
    if row['departamento'] in ['Chocó', 'Valle del Cauca']:
        color_pin = 'orange'
        estado = '⚠️ REVISIÓN TÉCNICA URGENTE'
    else:
        color_pin = 'green'
        estado = '✅ Operación Normal (Monitoreo)'

    popup_text = f"""
    <b>{row['nombre_activo']}</b><br>
    <b>Tipo:</b> {row['tipo']}<br>
    <b>Depto:</b> {row['departamento']}<br>
    <b>Estado:</b> {estado}
    """

    folium.CircleMarker(
        location=[row['latitud'], row['longitud']],
        radius=8,
        color=color_pin,
        fill=True,
        fill_color=color_pin,
        fill_opacity=0.9,
        popup=folium.Popup(popup_text, max_width=300)
    ).add_to(mapa_crisis)

# Mostrar el mapa interactivo directamente en Colab
mapa_crisis

--- ACTIVOS EXTRAÍDOS DE LA BASE DE DATOS SQL ---
  id_activo                 nombre_activo               tipo     departamento
0    ACT-01      Planta de Bombeo Istmina       Bombeo Crudo            Chocó
1    ACT-02             Subestación Yumbo  Energía Eléctrica  Valle del Cauca
2    ACT-03          Refinería de Cartago         Refinación  Valle del Cauca
3    ACT-04        Nodo Logístico Pereira     Almacenamiento        Risaralda
4    ACT-05  Planta Termoeléctrica Dorada         Generación           Caldas
5    ACT-06        Estación de Gas Yotoco          Gasoducto  Valle del Cauca
--------------------------------------------------
